In [1]:
!pip install gradio rembg pillow onnxruntime-gpu

  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl (11.0 MB)
   ---------------------------------------- 0.0/226.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/226.5 MB ? eta -:--:--
   ---------------------------------------- 2.6/226.5 MB 11.6 MB/s eta 0:00:20
    --------------------------------------- 5.5/226.5 MB 12.9 MB/s eta 0:00:18
   - -------------------------------------- 6.8/226.5 MB 10.2 MB/s eta 0:00:22
   - -------------------------------------- 9.4/226.5 MB 10.9 MB/s eta 0:00:20
   -- ------------------------------------- 11.8/226.5 MB 11.2 MB/s eta 0:00:20
   -- ------------------------------------- 13.6/226.5 MB 11.0 MB/s eta 0:00:20
   -- ------------------------------------- 16.0/226.5 MB 10.9 MB/s eta 0:00:20
   --- ------------------------------------ 18.9/226.5 MB 11.3 MB/s eta 0:00:19
   --- ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-common 1.5.0 requires psutil<7.2.0,>=5.7.3, but you have psutil 7.2.2 which is incompatible.
autogluon-core 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.8.0 which is incompatible.
autogluon-features 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.8.0 which is incompatible.
autogluon-tabular 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.8.0 which is incompatible.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.11.0+cu126 which is incompatible.
gluonts 0.16.2 requires numpy<2.2,>=1.16, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# filename: main_app.ipynb

# --------------------------------------------------
# 0. IMPORTS
# --------------------------------------------------

import os
from pathlib import Path
from PIL import Image
import gradio as gr
from rembg import remove

OUTPUT_DIR = Path("output_images")
OUTPUT_DIR.mkdir(exist_ok=True)

ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# --------------------------------------------------
# 1. HELPERS 
# --------------------------------------------------
def get_unique_output_path(input_path: Path, suffix: str = "_bez_pozadia") -> Path:
    ext = input_path.suffix.lower()
    stem = input_path.stem

    output_path = OUTPUT_DIR / f"{stem}{suffix}{ext}"
    counter = 1

    while output_path.exists():
        output_path = OUTPUT_DIR / f"{stem}{suffix}_{counter}{ext}"
        counter += 1

    return output_path


def remove_background_and_save(image_path: str):
    if not image_path:
        return None, None, "Nebol nahraty ziadny subor."

    input_path = Path(image_path)
    ext = input_path.suffix.lower()

    if ext not in ALLOWED_EXTENSIONS:
        return None, None, "Nepodporovany format. Pouzi JPG, JPEG alebo PNG."

    try:
        input_image = Image.open(input_path).convert("RGBA")

        output_image = remove(input_image)

        output_path = get_unique_output_path(input_path)

        if ext == ".png":
            output_image.save(output_path, format="PNG")
        # for JPEG instead of transparent (not supported), we use blank-white background
        elif ext in {".jpg", ".jpeg"}:
            white_bg = Image.new("RGB", output_image.size, (255, 255, 255))
            white_bg.paste(output_image, mask=output_image.split()[-1])
            white_bg.save(output_path, format="JPEG", quality=95)

        message = f"Hotovo. Vysledok bol ulozeny ako: {output_path.name}"
        return str(output_path), str(output_path), message

    except Exception as e:
        return None, None, f"Nastala chyba: {str(e)}"

# --------------------------------------------------
# 2. GRADIO UI
# --------------------------------------------------
with gr.Blocks(title="Odstranenie pozadia z obrazka") as demo:
    gr.Markdown("## Odstranenie pozadia z obrazka")
    gr.Markdown(
        "Nahraj obrazok vo formate **JPG, JPEG alebo PNG**. "
        "Aplikacia odstrani pozadie pomocou neuronovej siete a ulozi vysledok "
        "v rovnakom formate s upravenym nazvom suboru."
    )

    with gr.Row():
        input_image = gr.Image(
            type="filepath",
            label="Vstupny obrazok"
        )

    with gr.Row():
        process_btn = gr.Button("Odstranit pozadie")
        clear_btn = gr.Button("Vymazat")

    with gr.Row():
        output_preview = gr.Image(
            type="filepath",
            label="Nahlad vysledku"
        )

    with gr.Row():
        output_file = gr.File(label="Stiahnut vysledny subor")

    status_text = gr.Textbox(label="Stav", interactive=False)

    process_btn.click(
        fn=remove_background_and_save,
        inputs=[input_image],
        outputs=[output_preview, output_file, status_text]
    )

    clear_btn.click(
        fn=lambda: (None, None, None, ""),
        inputs=[],
        outputs=[input_image, output_preview, output_file, status_text]
    )

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
